# Fixed Mammography Training with ResNet-50 (INbreast Dataset)

**Improved version with fixes for common issues + stability improvements**

## Dataset: INbreast
- ~410 mammography images from 116 patients
- Binary classification: Benign vs Malignant (based on BI-RADS)
- Patient-wise train/val split (80/20)

## Key Fixes:
1. ✅ **ImageNet Normalization**: Properly normalize inputs for pretrained ResNet
2. ✅ **Class Weights**: Handle imbalanced dataset with weighted loss
3. ✅ **Better Hyperparameters**: Adjusted for small medical datasets
4. ✅ **Learning Rate Scheduler**: ReduceLROnPlateau to reduce metric oscillations
5. ✅ **Gradient Clipping**: Prevent unstable updates
6. ✅ **Metric Smoothing**: 3-epoch running average for stable early stopping
7. ✅ **Detailed Logging**: Track what the model is learning

---

## Configuration

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION - OPTIMIZED FOR SMALL MEDICAL DATASETS
# ============================================================================

# Paths - UPDATED FOR INBREAST
BASE_DIR = '/content/drive/MyDrive/INbreast'
PREPROCESSED_DIR = f'{BASE_DIR}/images'  # Preprocessed PNG images
METADATA_CSV = 'INbreast.csv'  # Single CSV file (will be split)
OUTPUT_DIR = '/content/drive/MyDrive/training_output_inbreast'

# Train/val split
TRAIN_VAL_SPLIT = 0.8  # 80% train, 20% validation
RANDOM_SEED = 42

# Model hyperparameters (OPTIMIZED FOR SMALL DATASETS)
LEARNING_RATE = 5e-5  # Lower for stability
WEIGHT_DECAY = 1e-5   # Lower to avoid over-regularization
DROPOUT_RATE = 0.2    # Lower for small datasets
UNFREEZE_FRACTION = 0.3  # Freeze more layers for small datasets

# Data augmentation (INCREASED for small datasets)
AUGMENTATION_STRENGTH = 0.8  # Higher augmentation to increase effective dataset size

# Training settings
BATCH_SIZE = 16  # Smaller batch for better generalization
MAX_EPOCHS = 150  # More epochs for slower learning rate
EARLY_STOPPING_PATIENCE = 25  # More patience
IMAGE_SIZE = (224, 224)

# Class weighting
USE_CLASS_WEIGHTS = True  # Handle imbalanced dataset

# Device
DEVICE = 'cuda'

print("✅ Configuration loaded")
print(f"\n💡 Using INbreast dataset")
print(f"   - Dataset: {METADATA_CSV}")
print(f"   - Train/Val split: {TRAIN_VAL_SPLIT*100:.0f}%/{(1-TRAIN_VAL_SPLIT)*100:.0f}%")
print(f"   - Lower learning rate and weight decay")
print(f"   - Less dropout")
print(f"   - More layers frozen")
print(f"   - Higher augmentation")
print(f"   - Class weights for imbalance")

## Step 1: Setup

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib pillow tqdm
print("✅ Dependencies installed")

## Step 2: Load Dataset with Class Weight Calculation

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# Load INbreast metadata
print("📊 Loading INbreast dataset metadata...")
df = pd.read_csv(METADATA_CSV)

print(f"Raw dataset: {len(df)} images")

# Create binary labels from Bi-RADS
# Benign (0): 1, 2, 3
# Malignant (1): 4a, 4b, 4c, 49, 5, 6
def create_label(birads):
    birads_str = str(birads).strip()
    if birads_str in ['1', '2', '3']:
        return 0
    elif birads_str in ['4a', '4b', '4c', '49', '5', '6']:
        return 1
    else:
        return None  # Unknown/invalid

df['label'] = df['Bi-Rads'].apply(create_label)

# Remove rows with invalid labels
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f"After label creation: {len(df)} images")
print(f"  Malignant (1): {(df['label']==1).sum()}")
print(f"  Benign (0): {(df['label']==0).sum()}")

# Create standardized columns
df['patient_id'] = df['Patient ID'].astype(str)
df['laterality'] = df['Laterality']
df['view_position'] = df['View']
df['breast_birads'] = 'BI-RADS ' + df['Bi-Rads'].astype(str)
df['image_id'] = df['File Name'].astype(int).astype(str)
df['breast_id'] = df['patient_id'] + '_' + df['laterality']

# Create image path (assuming PNG files in images directory)
df['image_path'] = df['image_id'] + '.png'

# Patient-wise train/val split (ensures no patient leakage)
print(f"\n🔀 Creating patient-wise train/val split...")
unique_patients = df['patient_id'].unique()
train_patients, val_patients = train_test_split(
    unique_patients,
    test_size=(1 - TRAIN_VAL_SPLIT),
    random_state=RANDOM_SEED
)

train_df = df[df['patient_id'].isin(train_patients)].copy()
val_df = df[df['patient_id'].isin(val_patients)].copy()

print(f"\n✅ Dataset split complete:")
print(f"  Train: {len(train_df)} images from {len(train_patients)} patients")
print(f"    Malignant: {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).sum()/len(train_df)*100:.1f}%)")
print(f"    Benign:    {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).sum()/len(train_df)*100:.1f}%)")
print(f"  Val: {len(val_df)} images from {len(val_patients)} patients")
print(f"    Malignant: {(val_df['label'] == 1).sum()} ({(val_df['label'] == 1).sum()/len(val_df)*100:.1f}%)")
print(f"    Benign:    {(val_df['label'] == 0).sum()} ({(val_df['label'] == 0).sum()/len(val_df)*100:.1f}%)")

# Count unique breasts
train_breasts = train_df['breast_id'].nunique()
val_breasts = val_df['breast_id'].nunique()
print(f"\n  Train breasts: {train_breasts}")
print(f"  Val breasts: {val_breasts}")

# Compute class weights
if USE_CLASS_WEIGHTS:
    labels = train_df['label'].values
    class_weights_array = compute_class_weight('balanced', classes=np.array([0, 1]), y=labels)
    pos_weight = class_weights_array[1] / class_weights_array[0]
    
    print(f"\n⚖️  Class Weights (for imbalanced data):")
    print(f"   Benign weight:    {class_weights_array[0]:.4f}")
    print(f"   Malignant weight: {class_weights_array[1]:.4f}")
    print(f"   Positive weight (malignant/benign): {pos_weight:.4f}")
    print(f"   This gives {pos_weight:.2f}x more weight to malignant samples")
else:
    pos_weight = 1.0
    print(f"\n⚠️  Not using class weights (may perform poorly on imbalanced data)")

## Dataset Visualization: Training & Validation Samples

Before training, let's visualize what our model will actually see.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*70)
print("DATASET VISUALIZATION")
print("="*70)

# Get sample images
train_mal = train_df[train_df['label'] == 1].head(4)
train_ben = train_df[train_df['label'] == 0].head(4)
val_mal = val_df[val_df['label'] == 1].head(4)
val_ben = val_df[val_df['label'] == 0].head(4)

# Create figure
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('DATASET SAMPLES: Training vs Validation', fontsize=18, fontweight='bold', y=0.995)

# Row labels
row_labels = [
    'TRAINING - Malignant',
    'TRAINING - Benign',
    'VALIDATION - Malignant',
    'VALIDATION - Benign'
]

datasets_to_plot = [train_mal, train_ben, val_mal, val_ben]
colors = ['red', 'green', 'red', 'green']

for row_idx, (dataset, label, color) in enumerate(zip(datasets_to_plot, row_labels, colors)):
    for col_idx, (_, row) in enumerate(dataset.iterrows()):
        ax = axes[row_idx, col_idx]
        
        # Load image
        img_path = os.path.join(PREPROCESSED_DIR, row['image_path'])
        
        if os.path.exists(img_path):
            img = Image.open(img_path)
            img_array = np.array(img)
            
            # Display image
            ax.imshow(img_array, cmap='gray')
            
            # Title with metadata
            birads = row.get('breast_birads', 'Unknown')
            laterality = row.get('laterality', 'Unknown')
            view = row.get('view_position', 'Unknown')
            
            title = f"{birads}\n{laterality} - {view}\n"
            title += f"Mean: {img_array.mean():.0f}"
            
            ax.set_title(title, fontsize=9, color=color, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'IMAGE\nNOT FOUND', ha='center', va='center',
                   fontsize=12, color='red', fontweight='bold')
            ax.set_title('MISSING', color='red')
        
        ax.axis('off')
        
        # Add row label on first column
        if col_idx == 0:
            ax.text(-0.1, 0.5, label, transform=ax.transAxes,
                   fontsize=12, fontweight='bold', va='center', ha='right',
                   rotation=90, color=color)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dataset_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Dataset Statistics:")
print(f"\nTraining Set:")
print(f"  Total: {len(train_df)} images")
print(f"  Malignant: {(train_df['label']==1).sum()} ({(train_df['label']==1).sum()/len(train_df)*100:.1f}%)")
print(f"  Benign:    {(train_df['label']==0).sum()} ({(train_df['label']==0).sum()/len(train_df)*100:.1f}%)")

print(f"\nValidation Set:")
print(f"  Total: {len(val_df)} images")
print(f"  Malignant: {(val_df['label']==1).sum()} ({(val_df['label']==1).sum()/len(val_df)*100:.1f}%)")
print(f"  Benign:    {(val_df['label']==0).sum()} ({(val_df['label']==0).sum()/len(val_df)*100:.1f}%)")

if 'breast_birads' in train_df.columns:
    print(f"\nBI-RADS Distribution in Training:")
    birads_dist = train_df['breast_birads'].value_counts().sort_index()
    for birads, count in birads_dist.items():
        print(f"  {birads}: {count} images ({count/len(train_df)*100:.1f}%)")
    
    print(f"\nBI-RADS Distribution in Validation:")
    birads_dist = val_df['breast_birads'].value_counts().sort_index()
    for birads, count in birads_dist.items():
        print(f"  {birads}: {count} images ({count/len(val_df)*100:.1f}%)")

print("\n⚠️  CHECK THE IMAGES ABOVE:")
print("   - Can you see breast tissue clearly?")
print("   - Are malignant and benign visually distinguishable?")
print("   - Are the images properly preprocessed?")
print("   - Do the BI-RADS categories match the labels?")

print("\n" + "="*70)
print("✅ Visualization complete - saved to:", os.path.join(OUTPUT_DIR, 'dataset_samples.png'))
print("="*70)

## Step 3: Define Model

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class ResNet50Classifier(nn.Module):
    def __init__(self, unfreeze_fraction=1.0, dropout_rate=0.0):
        super().__init__()
        self.unfreeze_fraction = max(0.0, min(1.0, unfreeze_fraction))
        self.dropout_rate = dropout_rate
        
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(2048, 1)
        
        self._setup_partial_finetuning()
    
    def _setup_partial_finetuning(self):
        all_layers = list(self.features.children())
        n_layers = len(all_layers)
        n_unfreeze = int(n_layers * self.unfreeze_fraction)
        
        for param in self.features.parameters():
            param.requires_grad = False
        
        if n_unfreeze > 0:
            layers_to_unfreeze = all_layers[-n_unfreeze:]
            for layer in layers_to_unfreeze:
                for param in layer.parameters():
                    param.requires_grad = True
        
        for param in self.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        features = self.features(x)
        features = self.avgpool(features)
        features = torch.flatten(features, 1)
        features = self.dropout(features)
        logits = self.classifier(features)
        probs = torch.sigmoid(logits).squeeze(1)
        return probs
    
    def get_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

model = ResNet50Classifier(
    unfreeze_fraction=UNFREEZE_FRACTION,
    dropout_rate=DROPOUT_RATE
).to(DEVICE)

print("✅ Model created:")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable parameters: {model.get_trainable_params():,}")
print(f"   Frozen parameters: {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}")
print(f"   Unfreeze fraction: {UNFREEZE_FRACTION:.1%}")
print(f"   Dropout rate: {DROPOUT_RATE:.1%}")

## Step 4: Create Data Loaders with ImageNet Normalization

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class MammogramDataset(Dataset):
    def __init__(self, metadata, image_dir, transform=None, augmentation=None):
        self.metadata = metadata.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.augmentation = augmentation
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        image_path = os.path.join(self.image_dir, row['image_path'])
        image = Image.open(image_path).convert('L')
        
        # Convert to tensor [0, 1]
        image = torch.from_numpy(np.array(image)).float() / 255.0
        image = image.unsqueeze(0)
        
        # Resize
        if self.transform is not None:
            image = self.transform(image)
        
        # Convert to 3-channel
        image = image.repeat(3, 1, 1)
        
        # Apply augmentation BEFORE normalization
        if self.augmentation is not None:
            image = self.augmentation(image)
        
        # ✅ KEY FIX: Apply ImageNet normalization
        # ResNet was pretrained with these values!
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        label = int(row['label'])
        image_id = row['image_id']
        
        return image, label, image_id

base_transform = transforms.Resize(IMAGE_SIZE)

class IntensityAugmentation:
    def __init__(self, strength=0.5):
        self.strength = strength
        self.brightness_factor = 0.2 * strength
        self.contrast_factor = 0.2 * strength
        self.noise_std = 0.05 * strength
    
    def __call__(self, x):
        if self.strength == 0.0:
            return x
        
        brightness_delta = torch.rand(1).item() * self.brightness_factor * 2 - self.brightness_factor
        x = x + brightness_delta
        
        contrast_delta = torch.rand(1).item() * self.contrast_factor * 2 - self.contrast_factor
        x = x * (1 + contrast_delta)
        
        noise = torch.randn_like(x) * self.noise_std
        x = x + noise
        
        x = torch.clamp(x, 0, 1)
        return x

train_augmentation = IntensityAugmentation(strength=AUGMENTATION_STRENGTH)

train_dataset = MammogramDataset(
    metadata=train_df,
    image_dir=PREPROCESSED_DIR,
    transform=base_transform,
    augmentation=train_augmentation
)

val_dataset = MammogramDataset(
    metadata=val_df,
    image_dir=PREPROCESSED_DIR,
    transform=base_transform,
    augmentation=None
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ Data loaders created with ImageNet normalization:")
print(f"   Mean: [0.485, 0.456, 0.406]")
print(f"   Std:  [0.229, 0.224, 0.225]")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Augmentation strength: {AUGMENTATION_STRENGTH:.1%}")

## Step 5: Training Components with Weighted Loss

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ✅ KEY FIX: Weighted loss for class imbalance
pos_weight_tensor = torch.tensor([pos_weight]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# Note: We need to change model to output logits instead of probabilities
# Let's modify the model's forward pass for training
class ResNet50ClassifierWithLogits(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
    
    def forward(self, x, return_logits=False):
        features = self.base_model.features(x)
        features = self.base_model.avgpool(features)
        features = torch.flatten(features, 1)
        features = self.base_model.dropout(features)
        logits = self.base_model.classifier(features).squeeze(1)
        
        if return_logits:
            return logits
        else:
            return torch.sigmoid(logits)

model_with_logits = ResNet50ClassifierWithLogits(model).to(DEVICE)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ✅ KEY FIX #1: Learning rate scheduler to reduce oscillations
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',           # Maximize PR-AUC
    factor=0.5,           # Reduce LR by half when plateau
    patience=5,           # Wait 5 epochs before reducing
    min_lr=1e-7
)

# ✅ KEY FIX #2: Gradient clipping value
MAX_GRAD_NORM = 1.0

class EarlyStopping:
    def __init__(self, patience=15, mode='max'):
        self.patience = patience
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False
        
        improved = (score > self.best_score) if self.mode == 'max' else (score < self.best_score)
        
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop

early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, mode='max')

def compute_metrics(predictions, labels):
    predictions = np.array(predictions)
    labels = np.array(labels)
    return {
        'auroc': roc_auc_score(labels, predictions),
        'pr_auc': average_precision_score(labels, predictions),
        'brier': brier_score_loss(labels, predictions)
    }

def aggregate_to_breast_level(image_predictions, metadata):
    breast_preds = {}
    breast_labels = {}
    
    for idx, row in metadata.iterrows():
        breast_id = row['breast_id']
        image_id = row['image_id']
        
        if image_id in image_predictions:
            pred = image_predictions[image_id]
            
            if breast_id not in breast_preds:
                breast_preds[breast_id] = []
                breast_labels[breast_id] = int(row['label'])
            
            breast_preds[breast_id].append(pred)
    
    final_preds = []
    final_labels = []
    
    for breast_id in breast_preds:
        preds = breast_preds[breast_id]
        noisy_or = 1.0 - np.prod([1.0 - p for p in preds])
        final_preds.append(noisy_or)
        final_labels.append(breast_labels[breast_id])
    
    return final_preds, final_labels

print("✅ Training components ready:")
print(f"   Loss: BCEWithLogitsLoss (with pos_weight={pos_weight:.4f})")
print(f"   Optimizer: AdamW")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   LR Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)")
print(f"   Gradient clipping: max_norm={MAX_GRAD_NORM}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Early stopping patience: {EARLY_STOPPING_PATIENCE}")

## Step 6: Training Loop with Detailed Logging

In [ ]:
from tqdm.notebook import tqdm
from collections import deque

history = {
    'train_loss': [],
    'val_pr_auc': [],
    'val_auroc': [],
    'val_brier': [],
    'train_predictions_mean': [],  # Track if model is learning
    'train_predictions_std': [],
    'learning_rate': []  # Track LR changes
}

# ✅ KEY FIX #3: Running average for metrics (reduces variance)
class MetricSmoother:
    def __init__(self, window_size=3):
        self.window_size = window_size
        self.values = deque(maxlen=window_size)
    
    def update(self, value):
        self.values.append(value)
        return sum(self.values) / len(self.values)

pr_auc_smoother = MetricSmoother(window_size=3)

os.makedirs(OUTPUT_DIR, exist_ok=True)
best_checkpoint_path = os.path.join(OUTPUT_DIR, 'best_model.pt')

print("🚀 Starting training...\n")
print("="*70)

for epoch in range(MAX_EPOCHS):
    # ========== TRAINING ==========
    model_with_logits.train()
    train_loss = 0.0
    n_batches = 0
    all_train_preds = []
    
    # Track LR before step
    current_lr = optimizer.param_groups[0]['lr']
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]")
    for images, labels, _ in pbar:
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE)
        
        # Forward (get logits for BCEWithLogitsLoss)
        logits = model_with_logits(images, return_logits=True)
        loss = criterion(logits, labels)
        
        # Track predictions
        with torch.no_grad():
            preds = torch.sigmoid(logits)
            all_train_preds.extend(preds.cpu().numpy())
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        
        # ✅ KEY FIX: Gradient clipping to prevent large updates
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        
        optimizer.step()
        
        train_loss += loss.item()
        n_batches += 1
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = train_loss / n_batches
    history['train_loss'].append(avg_train_loss)
    history['train_predictions_mean'].append(np.mean(all_train_preds))
    history['train_predictions_std'].append(np.std(all_train_preds))
    history['learning_rate'].append(current_lr)
    
    # ========== VALIDATION ==========
    model_with_logits.eval()
    image_predictions = {}
    image_labels = {}
    
    with torch.no_grad():
        for images, labels, image_ids in tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]  "):
            images = images.to(DEVICE)
            predictions = model_with_logits(images, return_logits=False).cpu().numpy()
            
            for img_id, pred, label in zip(image_ids, predictions, labels.numpy()):
                image_predictions[img_id] = float(pred)
                image_labels[img_id] = int(label)
    
    breast_predictions, breast_labels = aggregate_to_breast_level(image_predictions, val_df)
    val_metrics = compute_metrics(breast_predictions, breast_labels)
    
    # Raw metrics
    raw_pr_auc = val_metrics['pr_auc']
    history['val_pr_auc'].append(raw_pr_auc)
    history['val_auroc'].append(val_metrics['auroc'])
    history['val_brier'].append(val_metrics['brier'])
    
    # Smoothed metric for early stopping (reduces noise)
    smoothed_pr_auc = pr_auc_smoother.update(raw_pr_auc)
    
    # Print progress
    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Learning Rate: {current_lr:.2e}")
    print(f"  Train Pred Mean: {history['train_predictions_mean'][-1]:.4f} (should move from ~0.5)")
    print(f"  Train Pred Std:  {history['train_predictions_std'][-1]:.4f} (should increase if learning)")
    print(f"  Val PR-AUC (raw): {raw_pr_auc:.4f}")
    print(f"  Val PR-AUC (smoothed): {smoothed_pr_auc:.4f} ← used for early stopping")
    print(f"  Val AUROC:  {val_metrics['auroc']:.4f}")
    print(f"  Val Brier:  {val_metrics['brier']:.4f}")
    
    # ✅ KEY FIX: Step LR scheduler based on smoothed PR-AUC
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(smoothed_pr_auc)
    new_lr = optimizer.param_groups[0]['lr']
    
    # Print LR change if it happened
    if new_lr < old_lr:
        print(f"  📉 Learning rate reduced: {old_lr:.2e} → {new_lr:.2e}")
    
    # Save checkpoint if best (using smoothed metric)
    if early_stopping.best_score is None or smoothed_pr_auc > early_stopping.best_score:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'metrics': val_metrics,
            'smoothed_pr_auc': smoothed_pr_auc,
            'hyperparameters': {
                'learning_rate': LEARNING_RATE,
                'weight_decay': WEIGHT_DECAY,
                'dropout_rate': DROPOUT_RATE,
                'unfreeze_fraction': UNFREEZE_FRACTION,
                'augmentation_strength': AUGMENTATION_STRENGTH,
                'pos_weight': pos_weight
            }
        }, best_checkpoint_path)
        print(f"  💾 Saved best checkpoint (smoothed PR-AUC: {smoothed_pr_auc:.4f})")
    
    # Early stopping based on smoothed metric
    if early_stopping(smoothed_pr_auc, epoch):
        print(f"\n⚠️  Early stopping at epoch {epoch+1}")
        print(f"   Best epoch: {early_stopping.best_epoch + 1}")
        print(f"   Best smoothed PR-AUC: {early_stopping.best_score:.4f}")
        break
    
    print("="*70)

print("\n✅ Training complete!")
print(f"   Best checkpoint: {best_checkpoint_path}")
print(f"   Best epoch: {early_stopping.best_epoch + 1}")
print(f"   Best smoothed PR-AUC: {early_stopping.best_score:.4f}")

## Step 7: Training Diagnostics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
fig.suptitle('Training History and Diagnostics', fontsize=16, fontweight='bold')

# Training loss
axes[0, 0].plot(history['train_loss'], color='blue', linewidth=2)
axes[0, 0].set_title('Training Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('BCE Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best')
axes[0, 0].legend()

# Training predictions mean
axes[0, 1].plot(history['train_predictions_mean'], color='purple', linewidth=2)
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', label='Random (0.5)')
axes[0, 1].set_title('Train Prediction Mean (should diverge from 0.5)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Mean Prediction')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Training predictions std
axes[0, 2].plot(history['train_predictions_std'], color='orange', linewidth=2)
axes[0, 2].set_title('Train Prediction Std (should increase)', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Std of Predictions')
axes[0, 2].grid(True, alpha=0.3)

# Val PR-AUC (raw vs smoothed)
axes[1, 0].plot(history['val_pr_auc'], color='lightgreen', linewidth=1, alpha=0.5, label='Raw (noisy)')
# Compute smoothed for visualization
smoothed_pr_auc_viz = []
smoother_viz = MetricSmoother(window_size=3)
for val in history['val_pr_auc']:
    smoothed_pr_auc_viz.append(smoother_viz.update(val))
axes[1, 0].plot(smoothed_pr_auc_viz, color='green', linewidth=2, label='Smoothed (used)')
axes[1, 0].set_title('Validation PR-AUC: Raw vs Smoothed', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('PR-AUC')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best')
axes[1, 0].legend()

# Val AUROC
axes[1, 1].plot(history['val_auroc'], color='cyan', linewidth=2)
axes[1, 1].axhline(y=0.5, color='red', linestyle='--', label='Random')
axes[1, 1].set_title('Validation AUROC', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('AUROC')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axvline(early_stopping.best_epoch, color='red', linestyle='--')
axes[1, 1].legend()

# Val Brier
axes[1, 2].plot(history['val_brier'], color='magenta', linewidth=2)
axes[1, 2].set_title('Validation Brier Score', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Brier (lower better)')
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best')
axes[1, 2].legend()

# Learning rate (NEW)
axes[2, 0].plot(history['learning_rate'], color='brown', linewidth=2, marker='o')
axes[2, 0].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
axes[2, 0].set_xlabel('Epoch')
axes[2, 0].set_ylabel('Learning Rate')
axes[2, 0].set_yscale('log')
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best')
axes[2, 0].legend()

# Metric stability visualization (NEW)
if len(history['val_pr_auc']) > 5:
    # Compute rolling std of PR-AUC to show stability improvement
    window = 5
    rolling_std = []
    for i in range(len(history['val_pr_auc'])):
        if i < window:
            rolling_std.append(np.std(history['val_pr_auc'][:i+1]))
        else:
            rolling_std.append(np.std(history['val_pr_auc'][i-window+1:i+1]))
    
    axes[2, 1].plot(rolling_std, color='red', linewidth=2)
    axes[2, 1].set_title('PR-AUC Stability (rolling std, lower=more stable)', fontsize=12, fontweight='bold')
    axes[2, 1].set_xlabel('Epoch')
    axes[2, 1].set_ylabel('Rolling Std (window=5)')
    axes[2, 1].grid(True, alpha=0.3)

# Summary stats (NEW)
axes[2, 2].axis('off')
summary_text = f"""
TRAINING SUMMARY

Best Epoch: {early_stopping.best_epoch + 1}
Best PR-AUC: {early_stopping.best_score:.4f}
Final AUROC: {history['val_auroc'][early_stopping.best_epoch]:.4f}
Final Brier: {history['val_brier'][early_stopping.best_epoch]:.4f}

STABILITY METRICS:
PR-AUC Std (last 10): {np.std(history['val_pr_auc'][-10:]):.4f}
AUROC Std (last 10): {np.std(history['val_auroc'][-10:]):.4f}

CONVERGENCE:
LR Reductions: {sum(1 for i in range(1, len(history['learning_rate'])) if history['learning_rate'][i] < history['learning_rate'][i-1])}
Final LR: {history['learning_rate'][-1]:.2e}
"""
axes[2, 2].text(0.1, 0.9, summary_text, transform=axes[2, 2].transAxes,
               fontsize=11, verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Training history saved")
print("\n📊 Diagnostic checks:")
print(f"   ✓ Loss decreased: {history['train_loss'][0] > history['train_loss'][-1]}")
print(f"   ✓ Predictions diverged from 0.5: {abs(history['train_predictions_mean'][-1] - 0.5) > 0.05}")
print(f"   ✓ Prediction std increased: {history['train_predictions_std'][-1] > history['train_predictions_std'][0]}")
print(f"   ✓ AUROC > 0.7: {history['val_auroc'][early_stopping.best_epoch] > 0.7}")
print(f"\n📈 Stability improvements:")
print(f"   PR-AUC variance (last 10 epochs): {np.var(history['val_pr_auc'][-10:]):.6f}")
print(f"   AUROC variance (last 10 epochs): {np.var(history['val_auroc'][-10:]):.6f}")
print(f"   LR was reduced {sum(1 for i in range(1, len(history['learning_rate'])) if history['learning_rate'][i] < history['learning_rate'][i-1])} times")

## Summary

### Dataset: INbreast
- **Total images**: ~410 images (116 patients)
- **Labels**: Created from Bi-RADS scores
  - Benign (0): BI-RADS 1, 2, 3
  - Malignant (1): BI-RADS 4a, 4b, 4c, 5, 6
- **Split**: Patient-wise 80/20 train/val (prevents data leakage)
- **Validation set**: Small (~82 images, ~41 breasts) → high metric variance expected

### Key Improvements to Fix Unstable Metrics:

**Core Fixes:**
1. ✅ **ImageNet Normalization**: Properly normalizing with mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
2. ✅ **Class Weights**: Using BCEWithLogitsLoss with pos_weight to handle imbalance
3. ✅ **Better Regularization**: Lower dropout and weight decay for small datasets
4. ✅ **More Frozen Layers**: Only unfreezing 30% to prevent overfitting
5. ✅ **Higher Augmentation**: 80% strength to increase effective dataset size

**NEW Stability Fixes (addresses the oscillating metrics):**
6. ✅ **Learning Rate Scheduler**: ReduceLROnPlateau reduces LR when metrics plateau (reduces oscillations)
7. ✅ **Gradient Clipping**: Max norm=1.0 prevents large unstable updates
8. ✅ **Metric Smoothing**: 3-epoch running average for early stopping (filters out noise from small validation set)
9. ✅ **Enhanced Monitoring**: Track LR changes, metric stability, and convergence

### Why Metrics Were Unstable:
- **Very small validation set**: Only ~41 breasts → high variance (±15-20% swings are normal)
- **No LR scheduler**: Model "bounces" around optimal point at constant LR
- **No gradient clipping**: Occasional large gradients cause instability
- **Raw metrics for early stopping**: Noise amplified decision making

### Expected Improvements:
- **Much smoother metrics**: Oscillations should reduce from ±15% to ±5-7%
- **Better convergence**: LR will automatically reduce when plateauing
- **More stable checkpoints**: Using smoothed metrics prevents saving on lucky epochs
- **AUROC**: Target >0.70 for INbreast (smaller dataset)
- **PR-AUC**: Target >0.55 with reduced variance